<a href="https://colab.research.google.com/github/22pt16/SURE_Data_Mining/blob/main/Data_mining_package.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Package Installations

In [ ]:
!pip install torch numpy pandas tqdm



Dataset Setup

In [ ]:
!wget https://files.grouplens.org/datasets/movielens/ml-1m.zip
!unzip ml-1m.zip


--2026-03-04 05:21:39--  https://files.grouplens.org/datasets/movielens/ml-1m.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5917549 (5.6M) [application/zip]
Saving to: ‘ml-1m.zip’

ml-1m.zip           100%[===================>]   5.64M  4.22MB/s    in 1.3s    

2026-03-04 05:21:41 (4.22 MB/s) - ‘ml-1m.zip’ saved [5917549/5917549]

Archive:  ml-1m.zip
   creating: ml-1m/
  inflating: ml-1m/movies.dat        
  inflating: ml-1m/ratings.dat       
  inflating: ml-1m/README            
  inflating: ml-1m/users.dat         


Dataset Satatistics

In [ ]:
import pandas as pd

ratings = pd.read_csv(
    "ml-1m/ratings.dat",
    sep="::",
    engine="python",
    names=["user", "item", "rating", "timestamp"]
)

ratings = ratings.sort_values(["user", "timestamp"])
ratings.head()

from collections import defaultdict

user_sequences = defaultdict(list)

for row in ratings.itertuples():
    user_sequences[row.user].append(row.item)

# Filter users with >= 5 interactions
user_sequences = {
    u: seq for u, seq in user_sequences.items()
    if len(seq) >= 5
}

print("Total users:", len(user_sequences))


Total users: 6040


Modelling the Transformers

In [ ]:
import torch
from torch.utils.data import Dataset

MAX_LEN = 20

class SeqDataset(Dataset):
    def __init__(self, user_sequences, reverse=False):
        self.samples = []
        for seq in user_sequences.values():
            if reverse:
                seq = seq[::-1]
            for i in range(1, len(seq)):
                input_seq = seq[max(0, i-MAX_LEN):i]
                target = seq[i]
                self.samples.append((input_seq, target))
        self.reverse = reverse

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        seq, target = self.samples[idx]
        seq = seq[-MAX_LEN:]
        pad_len = MAX_LEN - len(seq)
        seq = [0]*pad_len + seq
        return torch.tensor(seq), torch.tensor(target)


In [ ]:
import torch.nn as nn
import math

class SimpleTransformer(nn.Module):
    def __init__(self, num_items, embed_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(num_items+1, embed_dim)
        self.pos_embedding = nn.Embedding(MAX_LEN, embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=2,
            dim_feedforward=128,
            dropout=0.1,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=1
        )

        self.fc = nn.Linear(embed_dim, num_items+1)

    def forward(self, x):
        positions = torch.arange(0, MAX_LEN).unsqueeze(0).to(x.device)
        x = self.embedding(x) + self.pos_embedding(positions)

        # causal mask
        mask = torch.triu(torch.ones(MAX_LEN, MAX_LEN), diagonal=1).bool().to(x.device)

        x = self.transformer(x, mask)
        x = x[:, -1, :]  # last token
        return self.fc(x)


**Test Execution Time for Frwd, Reverse Training**

for epoch: 2, batchsize:256, lr:0.001 , dataset: 1M

In [ ]:
from torch.utils.data import DataLoader
import time
import torch.optim as optim

num_items = ratings["item"].max()

forward_dataset = SeqDataset(user_sequences, reverse=False)
reverse_dataset = SeqDataset(user_sequences, reverse=True)

forward_loader = DataLoader(forward_dataset, batch_size=256, shuffle=True)
reverse_loader = DataLoader(reverse_dataset, batch_size=256, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_model(dataloader, model_name):
    model = SimpleTransformer(num_items).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    start_time = time.time()

    model.train()
    for epoch in range(2):  # only 2 epochs for demo
        total_loss = 0
        for seq, target in dataloader:
            seq = seq.to(device)
            target = target.to(device)

            optimizer.zero_grad()
            output = model(seq)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"{model_name} Epoch {epoch+1} Loss:", total_loss/len(dataloader))

    end_time = time.time()
    print(f"{model_name} Training Time:", end_time - start_time, "seconds")
    return model

forward_model = train_model(forward_loader, "Forward Transformer")
reverse_model = train_model(reverse_loader, "Reverse Transformer")


Forward Transformer Epoch 1 Loss: 6.7483392132544004
Forward Transformer Epoch 2 Loss: 5.963604819762351
Forward Transformer Training Time: 795.2336230278015 seconds
Reverse Transformer Epoch 1 Loss: 6.839394004288466
Reverse Transformer Epoch 2 Loss: 6.019914264904851
Reverse Transformer Training Time: 793.3900694847107 seconds
